In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import json
import numpy as np

# Read the CSV file
df = pd.read_csv('results/results-replicate-3.csv')

# Convert the hypervolumes and times from JSON strings to Python lists
df['Hypervolumes'] = df['Hypervolumes'].apply(json.loads)
df['Times'] = df['Times'].apply(json.loads)

# Get unique acquisition functions (excluding qLogEHVI as mentioned)
acq_functions = df['Acquisition'].unique()

# Color mapping for consistency across plots
color_map = {
    'Random': 'gray',
    'qLogNEHVI': 'blue',
    'qLogNParEGO': 'green',
    'qLogEHVI': 'red'  # Even though not present, including for completeness
}

def plot_average_cumulative_times_log_scale():
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for acq in acq_functions:
        # Skip qLogEHVI as mentioned
        if acq == 'qLogEHVI':
            continue
            
        acq_data = df[df['Acquisition'] == acq]
        
        if not acq_data.empty:
            # Find the maximum length of times across all runs
            max_length = max(len(times) for times in acq_data['Times'])
            
            # Skip zeroth time step by starting from index 1
            # Initialize arrays for average and std, skipping the first element
            avg_cumulative = np.zeros(max_length - 1)
            std_cumulative = np.zeros(max_length - 1)
            count_times = np.zeros(max_length - 1)
            
            # First pass: calculate cumulative times for each run and accumulate
            # Skip the zeroth time step (which is t=0)
            cumulative_times_all_runs = []
            for times in acq_data['Times']:
                # Skip the first element (t=0)
                if len(times) > 1:
                    cumulative = np.cumsum(times)[1:]  # Start from index 1
                    cumulative_times_all_runs.append(cumulative)
                    for i, t in enumerate(cumulative):
                        avg_cumulative[i] += t
                        count_times[i] += 1
            
            # Calculate average
            avg_cumulative = avg_cumulative / count_times
            
            # Second pass: calculate std dev
            for cumulative in cumulative_times_all_runs:
                for i, t in enumerate(cumulative):
                    if i < len(std_cumulative):
                        std_cumulative[i] += (t - avg_cumulative[i]) ** 2
            
            std_cumulative = np.sqrt(std_cumulative / count_times)
            
            # Create x-axis (iterations), starting from iteration 1
            iterations = np.arange(1, max_length)
            
            # Plot with error bands
            ax.plot(iterations, avg_cumulative, label=acq, color=color_map[acq])
            ax.fill_between(iterations, avg_cumulative - std_cumulative, 
                            avg_cumulative + std_cumulative, alpha=0.3, 
                            color=color_map[acq])
            
    ax.set_xticks(range(1,21))
    ax.set_xticklabels(range(1,21))
    
    # Set y-axis to log scale
    ax.set_yscale('log')
    
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Average Cumulative Runtime (seconds)')
    ax.legend()
    
    # Add grid lines that respect the log scale
    ax.grid(True, which="both", ls="-", alpha=0.2)
    
    plt.tight_layout()
    plt.savefig('plots/average_cumulative_runtime_log_scale.png', dpi=300)
    plt.close()

# Generate the log scale plot
plot_average_cumulative_times_log_scale()

print("Log scale runtime plot has been generated!")

Log scale runtime plot has been generated!


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import json
import numpy as np

# Read the CSV file
df = pd.read_csv('results/results-replicate-3.csv')

# Convert the hypervolumes and times from JSON strings to Python lists
df['Hypervolumes'] = df['Hypervolumes'].apply(json.loads)
df['Times'] = df['Times'].apply(json.loads)

# Get unique acquisition functions (excluding qLogEHVI as mentioned)
acq_functions = df['Acquisition'].unique()

# Color mapping for consistency across plots
color_map = {
    'Random': 'gray',
    'qLogNEHVI': 'blue',
    'qLogNParEGO': 'green',
    'qLogEHVI': 'red'  # Even though not present, including for completeness
}

# Function to plot hypervolumes for a specific run
def plot_run_hypervolumes(run_number):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for acq in acq_functions:
        # Skip qLogEHVI as mentioned it didn't finish
        if acq == 'qLogEHVI':
            continue
            
        run_data = df[(df['Run'] == run_number) & (df['Acquisition'] == acq)]
        
        if not run_data.empty:
            # Get the first row (should be only one per run/acquisition)
            row = run_data.iloc[0]
            
            # Create x-axis (iterations) - assuming hypervolumes length = iterations + 1 (including initial)
            iterations = np.arange(0, len(row['Hypervolumes']))
            
            # Plot
            ax.plot(iterations, row['Hypervolumes'], marker='o', label=acq, color=color_map[acq])
    
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Hypervolume')
    ax.set_title(f'Hypervolume vs. Iteration for Run {run_number}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'plots/hypervolume_run_{run_number}.png', dpi=300)
    plt.close()

# Function to plot average hypervolumes across all runs
def plot_average_hypervolumes():
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for acq in acq_functions:
        # Skip qLogEHVI as mentioned
        if acq == 'qLogEHVI':
            continue
            
        acq_data = df[df['Acquisition'] == acq]
        
        if not acq_data.empty:
            # Find the maximum length of hypervolumes across all runs
            max_length = max(len(hvs) for hvs in acq_data['Hypervolumes'])
            
            # Initialize arrays for average and std
            avg_hvs = np.zeros(max_length)
            std_hvs = np.zeros(max_length)
            count_hvs = np.zeros(max_length)
            
            # Sum up all values
            for hvs in acq_data['Hypervolumes']:
                for i, hv in enumerate(hvs):
                    avg_hvs[i] += hv
                    count_hvs[i] += 1
            
            # Calculate average
            avg_hvs = avg_hvs / count_hvs
            
            # Calculate std
            for hvs in acq_data['Hypervolumes']:
                for i, hv in enumerate(hvs):
                    std_hvs[i] += (hv - avg_hvs[i]) ** 2
            
            std_hvs = np.sqrt(std_hvs / count_hvs)
            
            # Create x-axis (iterations)
            iterations = np.arange(0, max_length)
            
            # Plot with error bands
            ax.plot(iterations, avg_hvs, label=acq, color=color_map[acq])
            ax.fill_between(iterations, avg_hvs - std_hvs, avg_hvs + std_hvs, 
                            alpha=0.3, color=color_map[acq])
            
    ax.set_xticks(range(21))
    ax.set_xticklabels(range(21))
    
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Average Hypervolume')
    ax.legend()
    
    # Add grid lines that respect the log scale
    ax.grid(True, which="both", ls="-", alpha=0.2)

    ax.legend()
    plt.tight_layout()
    plt.savefig('plots/average_hypervolume.png', dpi=300)
    plt.close()

# Function to create a comparison plot of the final hypervolume values
def plot_final_hypervolumes():
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Dictionary to store final hypervolumes for each acquisition function and run
    final_hvs = {}
    
    for acq in acq_functions:
        # Skip qLogEHVI as mentioned
        if acq == 'qLogEHVI':
            continue
            
        final_hvs[acq] = []
        
        for run in df['Run'].unique():
            run_data = df[(df['Run'] == run) & (df['Acquisition'] == acq)]
            
            if not run_data.empty:
                # Get last hypervolume value
                last_hv = run_data.iloc[0]['Hypervolumes'][-1]
                final_hvs[acq].append(last_hv)
    
    # Plot boxplots for final hypervolumes
    box_data = [final_hvs[acq] for acq in final_hvs.keys()]
    ax.boxplot(box_data, labels=list(final_hvs.keys()))
    
    ax.set_xlabel('Acquisition Function')
    ax.set_ylabel('Final Hypervolume')
    ax.set_title('Final Hypervolume Comparison Across Acquisition Functions')
    plt.tight_layout()
    plt.savefig('plots/final_hypervolume_comparison.png', dpi=300)
    plt.close()

# Plot each individual run
for run in df['Run'].unique():
    plot_run_hypervolumes(run)

# Plot average hypervolumes across all runs
plot_average_hypervolumes()

# Plot final hypervolumes comparison
plot_final_hypervolumes()

print("All plots have been generated!")

/tmp/ipykernel_2737425/2838737640.py:136: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(box_data, labels=list(final_hvs.keys()))


All plots have been generated!
